<a href="https://www.kaggle.com/code/avikdas567/cafa-6-protein-function-prediction?scriptVersionId=293862277" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# CAFA 6 Protein Function Prediction
# ============================================================

import pandas as pd
from tqdm import tqdm
from collections import defaultdict

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/cafa-6-protein-function-prediction"

TRAIN_TERMS = f"{DATA_ROOT}/Train/train_terms.tsv"
TRAIN_TAX  = f"{DATA_ROOT}/Train/train_taxonomy.tsv"
TEST_FASTA = f"{DATA_ROOT}/Test/testsuperset.fasta"
TEST_TAX   = f"{DATA_ROOT}/Test/testsuperset-taxon-list.tsv"

OUT_FILE = "submission.tsv"

# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------
TOP_K = {
    "F": 30,
    "P": 60,
    "C": 30
}

MIN_PROTEINS_PER_TAXON = 50   # below this → fallback to global
CHUNK_SIZE = 1000             # proteins per write chunk

# ------------------------------------------------------------
# Load training data
# ------------------------------------------------------------
print("Loading training annotations...")
train_terms = pd.read_csv(TRAIN_TERMS, sep="\t")
train_terms.columns = ["protein_id", "go_id", "aspect"]

train_tax = pd.read_csv(
    TRAIN_TAX,
    sep="\t",
    names=["protein_id", "taxon"]
)

train = train_terms.merge(train_tax, on="protein_id")

# ------------------------------------------------------------
# Load test taxonomy
# ------------------------------------------------------------
test_tax = pd.read_csv(
    TEST_TAX,
    sep="\t",
    names=["protein_id", "taxon"]
)

# ------------------------------------------------------------
# FAST FASTA ID reader
# ------------------------------------------------------------
def read_fasta_ids(path):
    with open(path) as f:
        return [line[1:].split()[0] for line in f if line.startswith(">")]

print("Reading test FASTA...")
test_ids = read_fasta_ids(TEST_FASTA)

test_df = (
    pd.DataFrame({"protein_id": test_ids})
    .merge(test_tax, on="protein_id", how="left")
)

print("Test proteins:", len(test_df))

# ------------------------------------------------------------
# GLOBAL GO PRIORS (fallback)
# ------------------------------------------------------------
print("Computing global priors...")
global_stats = (
    train
    .groupby(["aspect", "go_id"])
    .size()
    .reset_index(name="count")
)

global_stats["prob"] = (
    global_stats.groupby("aspect")["count"]
    .transform(lambda x: (x / x.max()) ** 0.5)
)

global_top = {}
for aspect, k in TOP_K.items():
    df = (
        global_stats[global_stats["aspect"] == aspect]
        .sort_values("prob", ascending=False)
        .head(k)
    )
    global_top[aspect] = df[["go_id", "prob"]].values.tolist()

# ------------------------------------------------------------
# TAXONOMY-CONDITIONED PRIORS
# ------------------------------------------------------------
print("Computing taxonomy-conditioned priors...")
taxon_counts = train.groupby("taxon")["protein_id"].nunique()

taxon_priors = defaultdict(dict)

for taxon, n_prot in tqdm(taxon_counts.items()):
    if n_prot < MIN_PROTEINS_PER_TAXON:
        continue

    subset = train[train["taxon"] == taxon]

    stats = (
        subset
        .groupby(["aspect", "go_id"])
        .size()
        .reset_index(name="count")
    )

    stats["prob"] = (
        stats.groupby("aspect")["count"]
        .transform(lambda x: (x / x.max()) ** 0.5)
    )

    for aspect, k in TOP_K.items():
        df = (
            stats[stats["aspect"] == aspect]
            .sort_values("prob", ascending=False)
            .head(k)
        )
        if len(df) > 0:
            taxon_priors[taxon][aspect] = df[["go_id", "prob"]].values.tolist()

print("Taxa with learned priors:", len(taxon_priors))

# ------------------------------------------------------------
# WRITE SUBMISSION
# ------------------------------------------------------------
print("Writing submission...")
with open(OUT_FILE, "w") as f_out:
    for i in tqdm(range(0, len(test_df), CHUNK_SIZE)):
        chunk = test_df.iloc[i:i + CHUNK_SIZE]

        rows = []
        for _, row in chunk.iterrows():
            pid = row["protein_id"]
            taxon = row["taxon"]

            for aspect in ["F", "P", "C"]:
                if taxon in taxon_priors and aspect in taxon_priors[taxon]:
                    terms = taxon_priors[taxon][aspect]
                else:
                    terms = global_top[aspect]

                for go_id, prob in terms:
                    rows.append([pid, go_id, f"{prob:.3f}"])

        pd.DataFrame(rows).to_csv(
            f_out,
            sep="\t",
            header=False,
            index=False
        )

print("\n Submission written:", OUT_FILE)

# Preview
print(pd.read_csv(OUT_FILE, sep="\t", header=None, nrows=10))

Loading training annotations...
Reading test FASTA...
Test proteins: 224309
Computing global priors...
Computing taxonomy-conditioned priors...


1381it [00:00, 2546.01it/s]


Taxa with learned priors: 47
Writing submission...


100%|██████████| 225/225 [01:15<00:00,  2.97it/s]


 Submission written: submission.tsv
            0           1      2
0  A0A0C5B5G6  GO:0005515  1.000
1  A0A0C5B5G6  GO:0042802  0.324
2  A0A0C5B5G6  GO:0042803  0.220
3  A0A0C5B5G6  GO:0003723  0.219
4  A0A0C5B5G6  GO:0003677  0.209
5  A0A0C5B5G6  GO:0003700  0.189
6  A0A0C5B5G6  GO:0043565  0.184
7  A0A0C5B5G6  GO:0003729  0.184
8  A0A0C5B5G6  GO:0000976  0.178
9  A0A0C5B5G6  GO:0000978  0.162
